# CSNAnalysis Tutorial 3: MFPTs and Hub Scores

**Dickson Lab, Michigan State University**

---

## Overview

This notebook demonstrates two kinetic analysis tools that go beyond equilibrium
populations and committors.

**Mean first passage time (MFPT):** The average number of steps needed to reach
a set of "sink" states when starting from state $i$. This is a natural measure of
the kinetic distance between states and basins.

**Hub scores:** A quantity $h_{A \to B}^C$ that measures the probability that a
transition from basin $A$ to basin $B$ passes through an intermediate community $C$
at least once.  High hub scores identify kinetic bottlenecks and obligate
intermediates (see [Dickson & Brooks, JCTC 2012](https://doi.org/10.1021/ct3003145)).

**This notebook covers:**
1. Computing MFPTs from all states to a set of sinks
2. Source-weighted average MFPTs
3. First passage time distributions (FPTDs)
4. Identifying candidate hub states from committor probabilities
5. Computing and interpreting hub scores

## Setup

In [ ]:
import numpy as np
import scipy.sparse
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline

from csnanalysis.csn import CSN
from csnanalysis.matrix import hubscores

count_mat = scipy.sparse.load_npz('matrix.npz')
our_csn = CSN(count_mat, symmetrize=True)
our_csn.trim(by_inflow=True, by_outflow=True, min_count=20)
_ = our_csn.calc_eig_weights()

Astates = [2031, 596, 1923, 3223, 2715]
Bstates = [1550, 3168, 476, 1616, 2590]
Ustates = list(np.loadtxt('state_U.dat', dtype=int))

print(f"Trimmed CSN: {our_csn.trim_nnodes} nodes")

## 1. Mean first passage times

`calc_mfpt` computes the MFPT from every state to a set of sink states.
Internally it uses exponential time-stepping: $T$ is repeatedly squared, so
the FPTD is sampled at checkpoints $1, 2, 4, 8, \ldots$ steps.  The MFPT is
then estimated as a weighted sum over these checkpoints.

The calculation stops when either `maxsteps` doublings have occurred or the
maximum un-absorbed probability falls below `tol`.

In [ ]:
# Use basin A states as sinks (only those that survived trimming)
sinks_A = [s for s in Astates if s in our_csn.trim_indices]
print(f"Basin A sinks in trimmed graph: {sinks_A}")

mfpt_A, fptd_A = our_csn.calc_mfpt(sinks=sinks_A, maxsteps=40, tol=1e-4)

nonzero_mfpt = mfpt_A[mfpt_A > 0]
print(f"\nMFPT to basin A (over non-sink, trimmed states):")
print(f"  Minimum : {nonzero_mfpt.min():.1f} steps")
print(f"  Median  : {np.median(nonzero_mfpt):.1f} steps")
print(f"  Maximum : {nonzero_mfpt.max():.1f} steps")

In [ ]:
# Distribution of MFPTs across all trimmed states
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nonzero_mfpt, bins=60, color='steelblue', edgecolor='none')
ax.axvline(nonzero_mfpt.mean(), color='red', ls='--', label=f'Mean = {nonzero_mfpt.mean():.0f}')
ax.set_xlabel("MFPT to basin A (lag-time steps)")
ax.set_ylabel("Number of states")
ax.set_title("Distribution of MFPTs to basin A")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Store MFPT as a node attribute for Gephi export
our_csn.add_trim_attr('mfpt_A', mfpt_A[our_csn.trim_indices], default=0)

## 2. Source-weighted average MFPT

It is often useful to ask: "what is the average MFPT from basin $B$ to basin $A$?"
The `sources` argument computes a weighted average over the source states using
their equilibrium populations as weights.

In [ ]:
sources_B = [s for s in Bstates if s in our_csn.trim_indices]

avg_mfpt_BA, avg_fptd_BA = our_csn.calc_mfpt(
    sinks=sinks_A, maxsteps=40, tol=1e-4, sources=sources_B
)
print(f"Average MFPT from basin B to basin A: {avg_mfpt_BA[0]:.1f} steps")

## 3. First passage time distributions

The FPTD gives the probability flux reaching the sink at each time checkpoint.
The checkpoints are at $2^1, 2^2, 4^2, \ldots$ steps (exponentially spaced), so
the x-axis is plotted on a log scale.

In [ ]:
# fptd_A has shape (n_checkpoints, n_states)
# bin midpoints: 2^(i-1) + 2^i for checkpoint i (0-indexed)
n_steps = fptd_A.shape[0]
bin_centers = np.array([2**(i-1) + 2**i for i in range(n_steps)])

# Pick a few example states: one close to basin A, one far
close_state  = sinks_A[0] if len(sinks_A) > 0 else our_csn.trim_indices[0]

# State with median MFPT (representative "average" kinetics)
trim_mfpts = mfpt_A[our_csn.trim_indices]
nonzero_mask = trim_mfpts > 0
median_val = np.median(trim_mfpts[nonzero_mask])
median_trim_idx = np.where(nonzero_mask)[0][np.argmin(np.abs(trim_mfpts[nonzero_mask] - median_val))]
median_state = our_csn.trim_indices[median_trim_idx]

# State with maximum MFPT
far_state = our_csn.trim_indices[int(np.argmax(mfpt_A[our_csn.trim_indices]))]

fig, ax = plt.subplots(figsize=(8, 4))
for state, label, color in [
    (median_state, f'Median MFPT state {median_state} ({mfpt_A[median_state]:.0f} steps)', 'steelblue'),
    (far_state,    f'Far state {far_state} ({mfpt_A[far_state]:.0f} steps)', 'coral'),
]:
    ax.plot(bin_centers, fptd_A[:, state], 'o-', label=label, color=color, ms=4)

ax.set_xscale('log')
ax.set_xlabel("Lag-time steps (log scale)")
ax.set_ylabel("Flux reaching basin A")
ax.set_title("First passage time distributions")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Hub scores

The **hub score** $h_{A \to B}^C$ is the probability that a trajectory that starts
in basin $A$ and ends in basin $B$ visits the intermediate community $C$ at least once.
A score near 1 means $C$ is an obligate kinetic intermediate; a score near 0 means
the $A \to B$ pathway bypasses $C$ entirely.

### Identifying candidate hub states

We identify candidate hub states as those in the "transition zone" between basins $A$
and $B$: states where both $p_A$ and $p_B$ are significant (neither basin dominates)
and $p_U$ is small.  These are the states through which $A \leftrightarrow B$
transitions most naturally pass.

In [ ]:
# Recompute committors (or load from notebook 2 if you saved them)
basins = [Astates, Bstates, Ustates]
comms = our_csn.calc_committors(basins, labels=['pA', 'pB', 'pU'])

# Transition-zone states: pA in [0.2, 0.8], pB > 0.1, pU < 0.7
transition_states = [
    i for i in our_csn.trim_indices
    if 0.2 <= comms[i, 0] <= 0.8
    and comms[i, 1] > 0.1
    and comms[i, 2] < 0.7
]
print(f"Transition-zone states: {len(transition_states)} found")

# Also define a "U-dominated" set for comparison
U_states_sample = [
    i for i in our_csn.trim_indices
    if comms[i, 2] > 0.8
][:len(transition_states)]   # same size for fair comparison
print(f"U-dominated states (for comparison): {len(U_states_sample)}")

### Computing hub scores

`hubscores` works directly on the trimmed transition matrix, so all state indices
must be expressed in the trimmed-index space.  Use `our_csn.idxs_to_trim()` for
the conversion.

> **Note:** The hub score calculation involves multiple matrix-squaring iterations
> on the full trimmed matrix and may take a minute for networks with $\sim$2000 nodes.

In [ ]:
# Convert everything to trimmed-matrix index space
trim_A    = our_csn.idxs_to_trim(Astates)
trim_B    = our_csn.idxs_to_trim(Bstates)
trim_hub  = our_csn.idxs_to_trim(transition_states)
trim_U    = our_csn.idxs_to_trim(U_states_sample)

print(f"Trim-space: A={len(trim_A)} states, B={len(trim_B)} states")
print(f"Candidate hubs (transition zone): {len(trim_hub)} states")
print(f"Candidate hubs (U-dominated):     {len(trim_U)} states")

In [ ]:
# Equilibrium weights in trimmed-matrix space
wts_eig = our_csn.calc_eig_weights()
trim_wts = wts_eig[our_csn.trim_indices]

# Hub scores for the transition-zone set
print("Computing hub scores for transition-zone states...")
h_transition = hubscores(
    our_csn.trim_transmat, trim_hub, [trim_A, trim_B], wts=trim_wts
)
print(f"  h(A→B) = {h_transition[0]:.4f}")
print(f"  h(B→A) = {h_transition[1]:.4f}")

In [ ]:
# Hub scores for the U-dominated set (expected to be lower)
if len(trim_U) > 0:
    print("Computing hub scores for U-dominated states...")
    h_U = hubscores(
        our_csn.trim_transmat, trim_U, [trim_A, trim_B], wts=trim_wts
    )
    print(f"  h(A→B) = {h_U[0]:.4f}")
    print(f"  h(B→A) = {h_U[1]:.4f}")
else:
    print("No U-dominated states available for comparison.")

In [ ]:
# Summary bar chart
labels_bar = ['Transition-zone\nhub states', 'U-dominated\nstates']
h_AB_vals  = [h_transition[0], h_U[0] if len(trim_U) > 0 else 0]
h_BA_vals  = [h_transition[1], h_U[1] if len(trim_U) > 0 else 0]

x = np.arange(len(labels_bar))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - 0.2, h_AB_vals, 0.35, label='h(A→B)', color='steelblue')
ax.bar(x + 0.2, h_BA_vals, 0.35, label='h(B→A)', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels_bar)
ax.set_ylabel("Hub score")
ax.set_ylim(0, 1.1)
ax.set_title("Hub scores: transition zone vs. U-dominated states")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Per-state hub score scan

Computing a hub score for each individual transition-zone state shows which
specific conformations are most obligately visited during $A \leftrightarrow B$
transitions.  These are the states most worth studying in detail.

In [ ]:
# Scan individual states (limit to top 20 by pA+pB to keep runtime manageable)
# Sort by pA + pB (most "between" A and B)
tz_sorted = sorted(
    transition_states,
    key=lambda s: comms[s, 0] + comms[s, 1],
    reverse=True
)[:20]

print(f"Scanning hub scores for top {len(tz_sorted)} transition-zone states...")
results = []
for s in tz_sorted:
    ts = our_csn.idxs_to_trim([s])
    if ts:
        sc = hubscores(our_csn.trim_transmat, ts, [trim_A, trim_B], wts=trim_wts)
        results.append((s, sc[0], sc[1]))

results.sort(key=lambda x: x[1], reverse=True)
print("\nTop 10 individual hub states (A→B direction):")
print(f"  {'State':>6}  {'h(A→B)':>8}  {'h(B→A)':>8}  {'pA':>6}  {'pB':>6}")
for s, hab, hba in results[:10]:
    print(f"  {s:6d}  {hab:8.4f}  {hba:8.4f}  {comms[s,0]:6.3f}  {comms[s,1]:6.3f}")

In [ ]:
if results:
    states_plot = [r[0] for r in results]
    h_AB_ind    = [r[1] for r in results]
    h_BA_ind    = [r[2] for r in results]
    pA_vals     = [comms[s, 0] for s in states_plot]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Bar chart of hub scores
    x = np.arange(len(states_plot))
    axes[0].bar(x - 0.2, h_AB_ind, 0.35, label='h(A→B)', color='steelblue')
    axes[0].bar(x + 0.2, h_BA_ind, 0.35, label='h(B→A)', color='coral')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([str(s) for s in states_plot], rotation=55, ha='right', fontsize=8)
    axes[0].set_ylabel("Hub score")
    axes[0].set_title("Individual hub scores")
    axes[0].legend()

    # Hub score vs. pA
    axes[1].scatter(pA_vals, h_AB_ind, color='steelblue', label='h(A→B)', zorder=3)
    axes[1].scatter(pA_vals, h_BA_ind, color='coral',     label='h(B→A)', zorder=3)
    axes[1].set_xlabel("pA")
    axes[1].set_ylabel("Hub score")
    axes[1].set_title("Hub score vs. committor pA")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Summary

| Quantity | What it measures | Key function |
|---|---|---|
| MFPT | Kinetic distance to a target basin | `CSN.calc_mfpt` |
| FPTD | Full first-passage time distribution | `CSN.calc_mfpt` (returns both) |
| Hub score | Obligatory-intermediate probability | `csnanalysis.matrix.hubscores` |

Together with the steady-state weights and committors from the earlier notebooks,
these tools give a comprehensive picture of the thermodynamics and kinetics encoded
in a molecular dynamics trajectory.